# Task #13: final training + single test run

Select a **T4 GPU runtime** (Runtime > Change runtime type), then run the cells in order.
All the logic is in `scripts/final_training.py`; `docs/final-model.md` explains the method.

1. Sweep: legal-bert + pos_weight, 2 learning rates x 5 folds, scored after epochs 1-3 (10 runs, ~3 h).
2. Select: pick the config and per-category thresholds from cross-validation only.
3. Final: train on all 408 training contracts (~25 min).
4. Test: score the official test split **once**. It locks after one run.

Everything is saved to Google Drive, and finished runs are skipped, so **after a disconnect,
just run everything again**. To share the sweep, each person sets `FOLDS` to different folds.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Break-Through-Tech/Accenture-1T-contract-review-challenge.git"
REPO_REF = "task-13-final-training"
REPO_DIR = Path("/content/contract-review")
DRIVE_DIR = Path("/content/drive/MyDrive/contract-review/final")  # survives disconnects

FOLDS = [0, 1, 2, 3, 4]          # sharing the sweep: each person lists different folds
LEARNING_RATES = [2e-5, 3e-5]    # fixed before running; see docs/final-model.md
EPOCHS = 3                       # each run is scored after epochs 1, 2 and 3
BATCH_SIZE = 8                   # 4 if the GPU runs out of memory; same value for every run
SMOKE_TEST = False               # True: a few steps everywhere to check setup; saved apart, never reported

ROOT = DRIVE_DIR / ("smoke" if SMOKE_TEST else "real")
RUNS_DIR = ROOT / "sweep"
FINAL_DIR = ROOT / "final_model"
RESULTS_DIR = ROOT / "results"   # copy this folder into docs/final-model-results/ when done

## 1. Drive, checkout, locked environment
Colab asks for permission to use Drive. An existing checkout is used as-is (no pull).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
for d in (RUNS_DIR, FINAL_DIR.parent, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
required = [
    "pyproject.toml", "uv.lock", "src/final_training.py", "scripts/final_training.py",
    "data/train_separate_questions.json", "data/test.json", "docs/tuned-baseline-results/folds.csv",
]
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Checkout is missing {missing}. Push the branch first.")
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
UV = [sys.executable, "-m", "uv"]
subprocess.run(UV + ["sync", "--locked", "--dev", "--python", "3.12"], cwd=REPO_DIR, check=True)

def run_project(*args):
    # relay output line by line so Colab shows it
    process = subprocess.Popen(
        UV + ["run", "--locked", "--no-sync", *map(str, args)], cwd=REPO_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)

def script(*args):
    run_project("python", "-B", "-m", "scripts.final_training", *args)

## 2. GPU and software checks
Stop here if either check fails.

In [ ]:
run_project("python", "-c", """
import torch, transformers
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, "Transformers:", transformers.__version__)
assert torch.cuda.is_available(), "No usable CUDA device in the project environment"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
""")
run_project("python", "-B", "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests/test_final_training.py")

## 3. Re-chunk + label train and test with legal-bert's tokenizer
About 3 minutes, repeated each session (written to the runtime disk). Expect train 14,023 chunks,
`longest_chunk_tokens` 512 for both splits, and `train_matches_task12: true`.

In [ ]:
script("prepare")

## 4. Sweep (about 3 hours on a T4)
Each learning rate finishes a fold before the next fold starts. A failed run is reported and the
loop continues. **Check `train_minutes` after the first run** to estimate the total.

In [ ]:
failed = []
smoke = ["--train-contracts", 6, "--validation-contracts", 2, "--max-steps", 4] if SMOKE_TEST else []
for fold in FOLDS:
    for lr in LEARNING_RATES:
        print(f"===== lr={lr} fold={fold} =====", flush=True)
        try:
            script("sweep", "--learning-rate", lr, "--validation-fold", fold,
                   "--output-dir", RUNS_DIR, "--epochs", EPOCHS, "--batch-size", BATCH_SIZE, *smoke)
        except subprocess.CalledProcessError as exc:
            failed.append((lr, fold))
            print(f"FAILED: lr={lr} fold={fold} ({exc}). See the traceback above.")
print("Failed runs:", failed or "none")

## 5. Select config + thresholds (CV only, no test data)
Needs all 5 folds for a learning rate. If teammates ran other folds, copy their `sweep` folders
into your Drive and add each one after `--runs-dir`.
Smoke mode skips this: smoke runs are refused on purpose.

In [ ]:
import json
import pandas as pd

if SMOKE_TEST:
    print("Smoke mode: selection skipped (smoke runs are never compared).")
else:
    script("select", "--runs-dir", RUNS_DIR, "--output-dir", RESULTS_DIR)
    selection = json.loads((RESULTS_DIR / "selection.json").read_text())
    print(json.dumps(selection["decision"], indent=2))
    print("Flagged:", selection["flagged_categories"] or "none")
    display(pd.read_csv(RESULTS_DIR / "sweep_summary.csv"))
    display(pd.read_csv(RESULTS_DIR / "thresholds.csv"))

## 6. Final model on all 408 training contracts (about 25 minutes)
Saved to Drive (`final_model/checkpoint`), not Git.

In [ ]:
if not SMOKE_TEST:
    script("final", "--selection", RESULTS_DIR / "selection.json", "--output-dir", FINAL_DIR)

## 7. ONE test run
Run this once, after the cells above succeeded. It refuses to run a second time.
Test predictions go to Drive; the metric tables go to `RESULTS_DIR`.

In [ ]:
if not SMOKE_TEST:
    script("test", "--checkpoint", FINAL_DIR / "checkpoint",
           "--selection", RESULTS_DIR / "selection.json",
           "--results-dir", RESULTS_DIR, "--output-dir", FINAL_DIR)
    display(pd.read_csv(RESULTS_DIR / "test_per_category.csv"))

## 8. Download the results for Git
Commit the unzipped files to `docs/final-model-results/`. Weights and predictions stay in Drive:
share the `final_model` folder with Thierno (task 14) and Stephanie (task 15).

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/final-model-results", "zip", root_dir=RESULTS_DIR)
files.download(archive)